In [1]:
from pathlib import Path
import json

from green_melon import YoloBbox, PascalBbox, WEEDMAIZE

In [3]:
IMAGES_DIR: Path = Path("data/weedmaize/images")
LABELS_DIR: Path = Path("data/weedmaize/labels")
YOLO_LABELS_DIR: Path = Path("data/weedmaize/yolo_labels")
YOLO_LABELS_DIR.mkdir(exist_ok=True)

backgrounds = []

for img in IMAGES_DIR.glob("*.JPG"):

    lbl = LABELS_DIR / (img.name + ".json")

    with lbl.open() as f:
        annots = json.load(f)

    image_width = annots["size"]["width"]
    image_height = annots["size"]["height"]

    yolo_lines = []
    for obj in annots["objects"]:

        target = obj["classTitle"]
        class_id = WEEDMAIZE[target]

        if class_id == -1: # background
            backgrounds.append(img.name)
            continue

        (xmin, ymin), (xmax, ymax) = obj["points"]["exterior"]

        pascal = PascalBbox(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax)
        yolo = pascal.to_yolo(class_id=class_id, image_width=image_width, image_height=image_height)
        yolo_lines.append(yolo.to_yolo_lines() + "\n")

    with (YOLO_LABELS_DIR / (img.stem + ".txt")).open("w") as f:
        f.writelines(yolo_lines)

with (YOLO_LABELS_DIR / "background_images.txt").open("w") as f:
    f.write("\n".join(backgrounds))